# Diffusion UI v3 — Geração de Imagens no Colab

Este notebook instala o **Stable Diffusion WebUI Forge** e a extensão **Cache Manager**.
Ele foi projetado para ser **extremamente robusto**, utilizando o `EnvironmentDoctor` para curar incompatibilidades de versão automaticamente.

### Como usar
1. Edite as opções na **Célula 0** abaixo (opcional).
2. Execute todas as células em ordem (`Runtime -> Run all`).

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🎛️ CÉLULA 0: CONFIGURAÇÃO (Edite antes de executar)
# ══════════════════════════════════════════════════════════════

CIVITAI_API_KEY = ""           # Cole sua API key do Civitai (opcional)
AUTO_DOWNLOAD_MODELS = False   # True = baixar modelos automaticamente no final
FORCE_FORGE_RECLONE = False    # True = deletar e baixar o Forge do zero
DEBUG_MODE = False             # True = mostrar logs detalhados do auto-healing

# Caminhos (geralmente não precisa alterar)
DRIVE_PATH = "/content/drive/MyDrive/Stable_Diffusion_Dados"
CACHE_PATH = "/content/cache"
FORGE_PATH = "/content/stable-diffusion-webui-forge"

print("✅ Configurações carregadas.")

## Célula 1: Upload de Extensões
Apenas execute esta célula se for a sua **primeira vez** ou se você atualizou os arquivos ZIP no seu computador.

In [ ]:
import os, zipfile, shutil
from google.colab import files

print("📦 ETAPA 1: UPLOAD DE EXTENSÕES")
print("═" * 60)
print("Se você não selecionar nenhum arquivo, esta etapa será pulada.")

try:
    uploaded = files.upload()
    if not uploaded:
        print("\n⏭️ Nenhum arquivo selecionado. Pulando upload.")
    else:
        # Monta drive temporariamente para salvar
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
            
        ext_dir = f"{DRIVE_PATH}/extensoes"
        os.makedirs(ext_dir, exist_ok=True)
        
        for filename in uploaded.keys():
            if filename.endswith('.zip'):
                print(f"\nExtraindo {filename}...")
                name = filename.replace('.zip', '')
                target = f"{ext_dir}/{name}"
                
                if os.path.exists(target):
                    shutil.rmtree(target)
                    
                with zipfile.ZipFile(filename, 'r') as zip_ref:
                    zip_ref.extractall(target)
                    
                print(f"✅ {filename} salvo no Drive: {target}")
                os.remove(filename)
                
except Exception as e:
    print(f"\n❌ Erro durante upload: {e}")

## Célula 2: Fundação
Monta o Google Drive e cria toda a estrutura de pastas (cache e persistência).

In [ ]:
import os, json

print("🏗️ ETAPA 2: FUNDAÇÃO (PASTAS E DRIVE)")
print("═" * 60)

# 1. Montar Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive montado com sucesso.")
except Exception as e:
    print(f"❌ Falha ao montar o Google Drive: {e}")

# 2. Criar pastas no Drive
drive_folders = ["Modelos_Base", "LoRAs", "VAEs", "Text_Encoders", "Imagens_Geradas", "logs"]
for folder in drive_folders:
    path = f"{DRIVE_PATH}/{folder}"
    os.makedirs(path, exist_ok=True)
print("✅ Estrutura de pastas do Drive pronta.")

# 3. Criar pastas de cache local
cache_folders = ["checkpoints", "loras", "vaes", "text_encoders"]
for folder in cache_folders:
    path = f"{CACHE_PATH}/{folder}"
    os.makedirs(path, exist_ok=True)
os.makedirs("/content/outputs_temp", exist_ok=True)
print("✅ Estrutura de pastas locais (cache) pronta.")

# 4. Salvar API Key (se fornecida)
if CIVITAI_API_KEY:
    config_path = f"{DRIVE_PATH}/.cache_config.json"
    try:
        config = {}
        if os.path.exists(config_path):
            with open(config_path, 'r') as f:
                config = json.load(f)
        config['api_key'] = CIVITAI_API_KEY
        with open(config_path, 'w') as f:
            json.dump(config, f)
        print("✅ API Key salva nas configurações.")
    except Exception as e:
        print(f"⚠️ Não foi possível salvar a API Key: {e}")

## Célula 3: Diagnóstico e Cura Automática (Auto-Healing)
Executa o **EnvironmentDoctor** para verificar e corrigir todas as 14 dependências conhecidas do Colab.

In [ ]:
import os, sys, shutil

print('💉 ETAPA 3: DIAGNÓSTICO E AUTO-HEALING')
print('═' * 60)

# Copiar lib/ do Drive (contém auto_healer.py e version_pins.py)
lib_src = f'{DRIVE_PATH}/lib'
lib_dst = '/content/lib'

try:
    if os.path.exists(lib_src):
        if os.path.exists(lib_dst):
            shutil.rmtree(lib_dst)
        shutil.copytree(lib_src, lib_dst)
        print('✅ Bibliotecas de sistema copiadas do Drive')
    else:
        print('❌ ERRO CRÍTICO: Pasta lib/ não encontrada no Drive!')
        print(f'   Esperado em: {lib_src}')
        print('   Por favor, faça upload do projeto inteiro para o Drive.')
        sys.exit(1)

    sys.path.insert(0, '/content')
    from lib.auto_healer import EnvironmentDoctor

    doctor = EnvironmentDoctor()

    )
    
    # Força reclonar Forge se configurado na Célula 0
    if FORCE_FORGE_RECLONE and os.path.exists(FORGE_PATH):
        print("⚠️ Removendo Forge existente devido à flag FORCE_FORGE_RECLONE...")
        shutil.rmtree(FORGE_PATH)

    print("\nIniciando diagnóstico (isso pode levar alguns minutos se precisar instalar Python ou pacotes):\n")
    
    def _progress(pct, name):
        print(f'\r  [{pct:>3}%] 💉 Vacinando: {name:<30}...', end='', flush=True)
        
    report = doctor.heal_all(progress_callback=_progress)
    
    print("\n")
    print(report.to_text())
    
    if report.critical_failures:
        print("\n⚠️ ALERTA: Houve falhas críticas.")
        print("O ambiente pode estar instável. O processo tentará continuar mesmo assim.")

except Exception as e:
    print(f'\n❌ Erro durante o auto-healing: {e}')
    print('Você pode tentar reiniciar o Runtime e executar novamente.')

## Célula 4: Instalar Extensões
Copia as extensões (Cache Manager, ReActor, etc.) para o Forge.

In [ ]:
import os, shutil

print('🧩 ETAPA 4: INSTALAR EXTENSÕES')
print('═' * 60)

forge_ext_dir = f"{FORGE_PATH}/extensions"
drive_ext_dir = f"{DRIVE_PATH}/extensoes"

if not os.path.exists(forge_ext_dir):
    print("⚠️ Forge não está instalado. Algo falhou na etapa anterior.")
elif not os.path.exists(drive_ext_dir):
    print(f"⚠️ Nenhuma extensão encontrada no Drive ({drive_ext_dir}).")
else:
    for ext in os.listdir(drive_ext_dir):
        src = os.path.join(drive_ext_dir, ext)
        dst = os.path.join(forge_ext_dir, ext)
        
        if os.path.isdir(src):
            print(f"Instalando {ext}...")
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"✅ {ext} instalado.")
            
print("\n✅ ETAPA 4 CONCLUÍDA")

## Célula 5: Central de Modelos
Interface gráfica para baixar modelos diretamente para o Google Drive via Civitai ou HuggingFace, além de upload do seu computador.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import subprocess
import time
from google.colab import files

pasta_modelos = f"{DRIVE_PATH}/Modelos_Base"
pasta_loras = f"{DRIVE_PATH}/LoRAs"
pasta_vaes = f"{DRIVE_PATH}/VAEs"
pasta_text_encoders = f"{DRIVE_PATH}/Text_Encoders"

caixa_api_key = widgets.Password(description='API Key:', placeholder='Sua chave do Civitai (opcional)', layout=widgets.Layout(width='400px'))
if CIVITAI_API_KEY:
    caixa_api_key.value = CIVITAI_API_KEY
api_box = widgets.HBox([caixa_api_key])

# ==========================================
# ABA 1: DOWNLOADER (INTERNET)
# ==========================================
lista_arquivos = widgets.VBox([])
def criar_linha_download():
    tipo = widgets.Dropdown(options=['Model / Checkpoint', 'LoRA', 'VAE', 'Text Encoder'], value='Model / Checkpoint', layout=widgets.Layout(width='180px'))
    link = widgets.Text(placeholder='Cole o link do download', layout=widgets.Layout(width='300px'))
    nome = widgets.Text(placeholder='ex: modelo.safetensors', layout=widgets.Layout(width='200px'))
    btn_remover = widgets.Button(description='❌', button_style='danger', layout=widgets.Layout(width='40px'))
    linha = widgets.HBox([tipo, link, nome, btn_remover])
    def remover(b): lista_arquivos.children = [c for c in lista_arquivos.children if c != linha]
    btn_remover.on_click(remover)
    return linha

lista_arquivos.children = [criar_linha_download()]
btn_adicionar = widgets.Button(description='➕ Adicionar Arquivo', button_style='info', layout=widgets.Layout(width='200px'))
btn_baixar = widgets.Button(description='⬇️ Baixar Tudo Agora', button_style='success', layout=widgets.Layout(width='200px'))
saida_log = widgets.Output()

def adicionar_linha(b): lista_arquivos.children = list(lista_arquivos.children) + [criar_linha_download()]
btn_adicionar.on_click(adicionar_linha)

def iniciar_download(b):
    with saida_log:
        clear_output()
        print("🚀 Iniciando processamento da fila...\n")
        chave_api = caixa_api_key.value.strip()
        for linha in lista_arquivos.children:
            tipo, url_original, nome = linha.children[0].value, linha.children[1].value.strip(), linha.children[2].value.strip()
            if not url_original or not nome: continue
            url_final = f"{url_original}{'&' if '?' in url_original else '?'}token={chave_api}" if chave_api and "civitai" in url_original.lower() else url_original

            if tipo == 'Model / Checkpoint': pasta_destino = pasta_modelos
            elif tipo == 'LoRA': pasta_destino = pasta_loras
            elif tipo == 'VAE': pasta_destino = pasta_vaes
            else: pasta_destino = pasta_text_encoders

            caminho_completo = f"{pasta_destino}/{nome}"
            print(f"⏳ Baixando: {nome}...")
            resultado = subprocess.run(['wget', '-q', '--show-progress', '-O', caminho_completo, url_final])
            if resultado.returncode == 0 and os.path.exists(caminho_completo) and os.path.getsize(caminho_completo) > 50000:
                print(f"✅ SUCESSO!\n")
            else:
                print(f"❌ ERRO no download. Verifique o link.\n")
                if os.path.exists(caminho_completo): os.remove(caminho_completo)
        print("🎉 FILA CONCLUÍDA!")
        atualizar_gerenciador()

btn_baixar.on_click(iniciar_download)
box_downloader = widgets.VBox([widgets.HTML("<p>Recomendado para Modelos Pesados (GBs). Velocidade máxima da rede do Google.</p>"), lista_arquivos, widgets.HBox([btn_adicionar, btn_baixar]), saida_log])

# ==========================================
# ABA 2: UPLOAD NATIVO DO COLAB
# ==========================================
dropdown_destino_upload = widgets.Dropdown(options=['Model / Checkpoint', 'LoRA', 'VAE', 'Text Encoder'], value='LoRA', description='Destino:')
btn_iniciar_upload = widgets.Button(description='🚀 Iniciar Uploader do Google', button_style='success', layout=widgets.Layout(width='250px'))
out_upload = widgets.Output()

def acionar_upload_nativo(b):
    with out_upload:
        clear_output()
        tipo = dropdown_destino_upload.value
        pasta = pasta_modelos if tipo == 'Model / Checkpoint' else pasta_loras if tipo == 'LoRA' else pasta_vaes if tipo == 'VAE' else pasta_text_encoders

        print(f"📡 Conectando ao uploader raiz do Google Colab...")
        print(f"➤ Destino escolhido: Pasta de {tipo}")
        print("👇 Clique no botão 'Escolher arquivos' que apareceu logo abaixo:")

        try:
            arquivos_upados = files.upload()
            if not arquivos_upados:
                print("\n⚠️ Nenhum arquivo foi selecionado. Operação cancelada.")
                return
            print(f"\n🚀 Gravando os arquivos direto no seu Google Drive...")
            for nome_arquivo, conteudo_bytes in arquivos_upados.items():
                caminho_final = os.path.join(pasta, nome_arquivo)
                with open(caminho_final, 'wb') as f:
                    f.write(conteudo_bytes)
                if os.path.exists(nome_arquivo): os.remove(nome_arquivo)
                print(f"   ✅ '{nome_arquivo}' salvo com sucesso!")
            print("\n🎉 Lote concluído!")
            atualizar_gerenciador()
        except Exception as e:
            print(f"\n❌ ERRO FATAL: {e}")

btn_iniciar_upload.on_click(acionar_upload_nativo)
aviso_upload = widgets.HTML("<div style='background-color: #d4edda; color: #155724; padding: 10px; border-radius: 5px; margin-bottom: 10px;'><b>✅ UPLOADER NATIVO:</b> O arquivo é gravado na memória e injetado diretamente no Drive.</div>")
box_upload = widgets.VBox([aviso_upload, widgets.HBox([dropdown_destino_upload, btn_iniciar_upload]), out_upload])

# ==========================================
# ABA 3: GERENCIADOR (CRUD)
# ==========================================
out_gerenciador = widgets.Output()
btn_atualizar_lista = widgets.Button(description='🔄 Atualizar Lista', button_style='info')

def listar_arquivos(pasta):
    if not os.path.exists(pasta): return []
    return [{'nome': f, 'caminho': os.path.join(pasta, f), 'tamanho': os.path.getsize(os.path.join(pasta, f)) / (1024 * 1024)} for f in os.listdir(pasta) if os.path.isfile(os.path.join(pasta, f))]

def atualizar_gerenciador(b=None):
    with out_gerenciador:
        clear_output()
        display(widgets.HTML("<p>Seus arquivos atuais no Google Drive:</p>"))
        for titulo, pasta in [("Modelos Base", pasta_modelos), ("LoRAs", pasta_loras), ("VAEs", pasta_vaes), ("Text Encoders", pasta_text_encoders)]:
            display(widgets.HTML(f"<h4>{titulo}</h4>"))
            arquivos = listar_arquivos(pasta)
            if not arquivos:
                display(widgets.HTML("<i>Nenhum arquivo encontrado.</i>"))
                continue
            for arq in arquivos:
                lbl_nome = widgets.Label(f"📄 {arq['nome']} ({arq['tamanho']:.1f} MB)", layout=widgets.Layout(width='320px'))
                txt_link_sub = widgets.Text(placeholder='Cole novo link', layout=widgets.Layout(width='180px'))
                btn_sub = widgets.Button(description='🔄 Substituir', button_style='warning', layout=widgets.Layout(width='100px'))
                btn_del = widgets.Button(description='🗑️ Apagar', button_style='danger', layout=widgets.Layout(width='90px'))
                box_confirmacao = widgets.HBox([])
                linha = widgets.HBox([lbl_nome, txt_link_sub, btn_sub, btn_del, box_confirmacao])

                def criar_evento_apagar(path, box, btn_d, btn_s):
                    def ao_clicar_apagar(b):
                        btn_sim, btn_nao = widgets.Button(description='✅ Confirmar', button_style='success', layout=widgets.Layout(width='100px')), widgets.Button(description='❌ Cancelar', button_style='danger', layout=widgets.Layout(width='90px'))
                        def ao_confirmar(bs): os.remove(path); atualizar_gerenciador()
                        def ao_cancelar(bn): box.children = []; btn_d.disabled = False; btn_s.disabled = False
                        btn_sim.on_click(ao_confirmar); btn_nao.on_click(ao_cancelar)
                        btn_d.disabled, btn_s.disabled = True, True
                        box.children = [widgets.Label("Certeza?"), btn_sim, btn_nao]
                    return ao_clicar_apagar
                btn_del.on_click(criar_evento_apagar(arq['caminho'], box_confirmacao, btn_del, btn_sub))

                def criar_evento_substituir(path, txt, nome):
                    def ao_clicar_substituir(b):
                        url = txt.value.strip()
                        if not url: return
                        with out_gerenciador:
                            clear_output()
                            chave = caixa_api_key.value.strip()
                            url_final = f"{url}{'&' if '?' in url else '?'}token={chave}" if chave and "civitai" in url.lower() else url
                            print("🗑️ 1. Apagando antigo..."); os.remove(path)
                            print("⏳ 2. Baixando novo...")
                            subprocess.run(['wget', '-q', '--show-progress', '-O', path, url_final])
                            time.sleep(2); atualizar_gerenciador()
                    return ao_clicar_substituir
                btn_sub.on_click(criar_evento_substituir(arq['caminho'], txt_link_sub, arq['nome']))
                display(linha)

btn_atualizar_lista.on_click(atualizar_gerenciador)
atualizar_gerenciador()

abas = widgets.Tab(children=[box_downloader, box_upload, widgets.VBox([btn_atualizar_lista, out_gerenciador])])
abas.set_title(0, '📥 Baixar (Internet)')
abas.set_title(1, '📤 Upload (PC Local)')
abas.set_title(2, '📁 Gerenciar Arquivos')
display(widgets.HTML("<h2>🎛️ Central de Modelos</h2>"), api_box, abas)


## Célula 6: Inicializar o Forge
Tudo pronto. Esta célula inicia a interface Web com um link público.

In [ ]:
import os

print('🚀 ETAPA 6: INICIANDO O FORGE')
print('═' * 50)
print('⏳ Aguarde o link público terminar em .gradio.live ...')
print()

# Previne que matplotlib tente abrir janelas nativas
os.environ['MPLBACKEND'] = 'agg'

# Certifica que as restrições de versão estão ativas para o pip interno do Forge
os.environ['PIP_CONSTRAINT'] = '/content/pip_constraints_sd.txt'

%cd {FORGE_PATH}

args = "--share --enable-insecure-extension-access --theme dark --listen"

# Se estamos no CPU (detectado pelo Doctor), força args de CPU
import torch
if not torch.cuda.is_available():
    args += " --use-cpu all --skip-torch-cuda-test --no-half --precision full"
    print("⚠️ Modo CPU ativado (lento).")

!python3.10 launch.py {args}